In [2]:
macro_data_path="../../../datasets/macro_raw"
# read raw macro source files from macro_data_path
import os
import pandas as pd
import numpy as np
# some files are csv and some are txt
# read all files in the directory
def read_files_from_path(path):
    files = os.listdir(path)
    data = {}
    for file in files:
        print(file)
        if file.endswith(".csv"):
            data[file] = pd.read_csv(os.path.join(path, file))
        elif file.endswith(".txt"):
            data[file] = pd.read_table(os.path.join(path, file))
    return data
# print the first 5 rows of each file
data = read_files_from_path(macro_data_path)
for file in data:
    print(file)
    print(data[file].head(1))
    print("\n\n")

macro1_Quarterly.txt
ebp_csv.csv
fci_g_public_monthly_3yr.csv
macro1_Monthly.txt
FED_Note_Term_SOFR.csv
DKW_updates.csv
FEDS-Note-2873-cie-data.csv
lmci_feds.csv
macro1_Quarterly.txt
         DATE  A191RL1Q225SBEA  A191RP1Q027SBEA
0  1947-04-01             -1.0              4.7



ebp_csv.csv
         date  gz_spread       ebp  est_prob
0  1973-01-01   1.130771 -0.044934  0.187934



fci_g_public_monthly_3yr.csv
         date  FCI-G Index (baseline)       FFR  10Yr Treasury  Mortgage Rate  \
0  1990-01-31                 0.05408 -0.028153      -0.019422      -0.115932   

        BBB  Stock Market  House Prices    Dollar  
0  0.028884     -0.245136     -0.112233  0.546072  



macro1_Monthly.txt
         DATE  CPIAUCSL  FEDFUNDS  GS10  M1NS  M2SL  T10YIEM  UNRATE
0  1947-01-01     21.48       NaN   NaN   NaN   NaN      NaN     NaN



FED_Note_Term_SOFR.csv
         DATE  REALIZED_1M  REALIZED_3M  REALIZED_6M
0  1998-03-01     5.499957          NaN          NaN



DKW_updates.csv
      

In [3]:
# unify the column names
# change "date","DATE","period" to "date"

for file in data:
    if "date" in data[file].columns:
        data[file] = data[file].rename(columns={"date":"date"})
    elif "DATE" in data[file].columns:
        data[file] = data[file].rename(columns={"DATE":"date"})
    elif "period" in data[file].columns:
        data[file] = data[file].rename(columns={"period":"date"})
    else:
        print("no date column in {}".format(file))

import pandas as pd

# List of possible date formats
date_formats = [
    "%Y-%m-%d",    # 1999-3-31, 2000-01-01
    "%d-%m-%Y",    # 05-12-2021
    "%Ym%m",       # 1976m8
]

# Iterate through the files
for file in data:
    if "date" in data[file].columns:
        date_column = data[file]["date"]
        converted_date = None
        print(f"Converting dates in {file}")
        print("date before conversion",date_column.head(5))

        # Try each format
        for fmt in date_formats:
            try:
                converted_date = pd.to_datetime(date_column, format=fmt, errors='coerce')
                # Break if parsing succeeds (i.e., no more NaT values)
                if not converted_date.isna().all():
                    break
            except Exception as e:
                continue

        # Fallback to automatic parsing for unhandled formats
        if converted_date is None or converted_date.isna().any():
            try:
                converted_date = pd.to_datetime(date_column, errors='coerce')
            except Exception as e:
                print(f"Error parsing dates in file {file}: {e}")

        # Check for remaining NaT values and log them
        if converted_date.isna().any():
            print(f"Date format error in {file}: ", data[file].loc[converted_date.isna(), "date"].values)

        # Retain only month and year
        data[file]["date"] = converted_date.dt.to_period('M').dt.to_timestamp()
        print(f"Converted dates in {file} to {converted_date.dt.to_period('M').dt.to_timestamp().dt.strftime('%Y-%m')}")

    else:
        print(f"No date column in {file}")

    

Converting dates in macro1_Quarterly.txt
date before conversion 0    1947-04-01
1    1947-07-01
2    1947-10-01
3    1948-01-01
4    1948-04-01
Name: date, dtype: object
Converted dates in macro1_Quarterly.txt to 0      1947-04
1      1947-07
2      1947-10
3      1948-01
4      1948-04
        ...   
310    2024-10
311    2025-01
312    2025-04
313    2025-07
314    2025-10
Name: date, Length: 315, dtype: object
Converting dates in ebp_csv.csv
date before conversion 0    1973-01-01
1    1973-02-01
2    1973-03-01
3    1973-04-01
4    1973-05-01
Name: date, dtype: object
Converted dates in ebp_csv.csv to 0      1973-01
1      1973-02
2      1973-03
3      1973-04
4      1973-05
        ...   
634    2025-11
635    2025-12
636    2026-01
637    2026-02
638    2026-03
Name: date, Length: 639, dtype: object
Converting dates in fci_g_public_monthly_3yr.csv
date before conversion 0    1990-01-31
1    1990-02-28
2    1990-03-30
3    1990-04-30
4    1990-05-31
Name: date, dtype: object
Conver

In [4]:
# cast all non-date columns to numeric
for file in data:
    for column in data[file].columns:
        if column != "date":
            data[file][column] = pd.to_numeric(data[file][column], errors='coerce')

In [5]:
# group mean by date
for file in data:
    data[file] = data[file].groupby("date").mean()
    # print(data[file].head(5))
    # still use date as a column
    data[file].reset_index(inplace=True)

In [6]:
# merge all data by date column
merged_data = None
for file in data:
    if merged_data is None:
        merged_data = data[file]
    else:
        merged_data = pd.merge(merged_data, data[file], on="date", how="outer")

In [7]:
# select date range from 2000-01 to 2024-12
merged_data = merged_data[(merged_data["date"] >= "2000-01") & (merged_data["date"] <= "2024-12")]

In [8]:
# describe the data and check nan values of each column
# Use forward fill only. Do not backfill from future observations.
merged_data = merged_data.sort_values("date").reset_index(drop=True)

selected_macro_path = "macro_list.txt"
with open(selected_macro_path, "r") as f:
    Macro_features = [line.strip() for line in f if line.strip()]

missing_features = [feature for feature in Macro_features if feature not in merged_data.columns]
if missing_features:
    raise KeyError(f"Selected macro features missing from merged macro data: {missing_features}")

merged_data = merged_data.ffill()

complete_selected_rows = merged_data[Macro_features].notna().all(axis=1)
if not complete_selected_rows.any():
    missing_counts = merged_data[Macro_features].isna().sum().sort_values(ascending=False)
    raise ValueError(
        "No date has full selected macro coverage after forward fill. "
        f"Missing counts:\n{missing_counts[missing_counts > 0]}"
    )

first_complete_idx = complete_selected_rows.idxmax()
if first_complete_idx > 0:
    old_start = merged_data.loc[0, "date"]
    new_start = merged_data.loc[first_complete_idx, "date"]
    print(f"Dropping leading macro rows without full selected coverage: {old_start.date()} -> {new_start.date()}")
    print("Columns causing the later start:")
    first_valid_dates = merged_data[Macro_features].apply(lambda col: merged_data.loc[col.notna(), "date"].min())
    print(first_valid_dates.sort_values().tail(10))

merged_data = merged_data.loc[first_complete_idx:].reset_index(drop=True)
remaining_missing = merged_data[Macro_features].isna().sum()
if remaining_missing.any():
    raise ValueError(f"Selected macro data still has NaNs after trimming and forward fill:\n{remaining_missing[remaining_missing > 0]}")

print("macro date range after no-leak coverage trim", merged_data["date"].min(), merged_data["date"].max())
print("selected macro feature count", len(Macro_features))
print(merged_data.describe())

Dropping leading macro rows without full selected coverage: 2000-01-01 -> 2000-03-01
Columns causing the later start:
ic.raw.10                  2000-01-01
ic.fitted.10               2000-01-01
exp.real.short.rate.5f5    2000-01-01
exp.inflation.5f5          2000-01-01
tips.liq.prem.5f5          2000-01-01
nominal.yield.raw.5f5      2000-01-01
nominal.yield.fitted.5f5   2000-01-01
tips.liq.prem.10           2000-01-01
CIE_spf                    2000-03-01
CIE_mich                   2000-03-01
dtype: datetime64[ns]
macro date range after no-leak coverage trim 2000-03-01 00:00:00 2024-12-01 00:00:00
selected macro feature count 46
                                date  A191RL1Q225SBEA  A191RP1Q027SBEA  \
count                            298       298.000000       298.000000   
mean   2012-07-16 12:38:39.463087360         2.287248         4.676174   
min              2000-03-01 00:00:00       -28.000000       -29.100000   
25%              2006-05-08 18:00:00         1.125000         3.150

In [9]:
# save the merged data to a csv file
data_folder = "../../../datasets/macro_processed"
# change the 'date' column to "Date"
merged_data = merged_data.rename(columns={"date":"Date"})
# create the folder if it does not exist
if not os.path.exists(data_folder):
    os.makedirs(data_folder)
merged_data.to_csv(os.path.join(data_folder, "macro_data.csv"), index=False)

In [10]:
# read ETF stock data from the raw Yahoo Finance folder and merge it
import os
import pandas as pd

stock_data_path = "../../../workdir/yahoofinance_day_prices_future_etfs"
asset_list_path = "../../../configs/_asset_list_/future_etfs.txt"
output_file_name = "merged_future_etfs_stock_data.csv"

with open(asset_list_path, "r") as f:
    etf_tickers = [line.strip() for line in f if line.strip() and not line.strip().startswith("#")]

required_columns = ["Date", "Open", "High", "Low", "Close", "Adj Close", "Volume"]
required = set(required_columns)

def read_files_from_path(path, tickers):
    data = {}
    missing_files = []
    for ticker in tickers:
        file = f"{ticker}.csv"
        filepath = os.path.join(path, file)
        if not os.path.isfile(filepath):
            missing_files.append(filepath)
            continue
        try:
            df = pd.read_csv(filepath)
        except pd.errors.EmptyDataError:
            print(f"Skipping empty file: {file}")
            continue
        if len(df) == 0:
            print(f"Skipping file with no rows: {file}")
            continue
        if not required.issubset(df.columns):
            print(f"Skipping non-OHLCV file: {file}")
            continue
        # yfinance multi-ticker CSVs can carry a second row with ticker labels
        # under each OHLCV column and a blank Date. Drop that metadata row.
        if pd.isna(df.loc[0, "Date"]) or str(df.loc[0, "Date"]).strip() == "":
            df = df.iloc[1:].copy()
        data[ticker] = df
    if missing_files:
        raise FileNotFoundError("Missing ETF raw CSVs:\n" + "\n".join(missing_files))
    return data

data = read_files_from_path(stock_data_path, etf_tickers)
print(f"Loaded {len(data)} ETFs: {sorted(data.keys())}")

# concatenate all ETF data together into one dataframe
stock_data = None
for ticker in etf_tickers:
    df = data[ticker].copy()
    df = df[required_columns].copy()
    df["Date"] = pd.to_datetime(df["Date"], utc=True, errors="coerce").dt.tz_localize(None)
    for col in ["Open", "High", "Low", "Close", "Adj Close", "Volume"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.dropna(subset=required_columns)
    df["Date"] = df["Date"].dt.strftime("%Y-%m-%d")
    df["ticker"] = ticker
    if stock_data is None:
        stock_data = df
    else:
        stock_data = pd.concat([stock_data, df], axis=0)

unique_dates = stock_data["Date"].unique()
print("unique dates", len(unique_dates))
print(stock_data.columns)
print(stock_data.head(5))
print("date range of stock data", stock_data["Date"].min(), stock_data["Date"].max())
print("size of stock data", stock_data.shape)
print("unique tickers", stock_data["ticker"].unique())
print(stock_data.isna().sum())
print(stock_data.groupby("ticker")["Date"].agg(["min", "max", "count"]))

# save the merged ETF stock data to a csv file
stock_data = stock_data.sort_values(["ticker", "Date"]).reset_index(drop=True)
data_folder = "../../../datasets/stock_data"
if not os.path.exists(data_folder):
    os.makedirs(data_folder)
stock_data.to_csv(os.path.join(data_folder, output_file_name), index=False)

Loaded 18 ETFs: ['CORN', 'CYB', 'DBB', 'DBC', 'FXA', 'FXB', 'FXC', 'FXE', 'FXY', 'GLD', 'IWM', 'NIB', 'QQQ', 'SLV', 'SPY', 'UGA', 'UNG', 'USO']
unique dates 6037
Index(['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume',
       'ticker'],
      dtype='object')
         Date         Open         High          Low        Close  \
1  2007-04-18  6451.200195  6517.759766  6403.839844  6498.560059   
2  2007-04-19  6438.399902  6513.919922  6412.799805  6471.680176   
3  2007-04-20  6411.520020  6411.520020  6336.000000  6341.120117   
4  2007-04-23  6344.959961  6553.600098  6297.600098  6528.000000   
5  2007-04-24  6630.399902  6630.399902  6460.160156  6539.520020   

     Adj Close  Volume ticker  
1  6498.560059   327.0    UNG  
2  6471.680176   742.0    UNG  
3  6341.120117   524.0    UNG  
4  6528.000000  1067.0    UNG  
5  6539.520020   998.0    UNG  
date range of stock data 2000-01-03 2023-12-29
size of stock data (81697, 8)
unique tickers ['UNG' 'GLD' 'SLV' 'SPY' 'QQQ

In [11]:
import os
import pandas as pd

# load the merged ETF stock data and resample the selected macro data
stock_data_path = "../../../datasets/stock_data/merged_future_etfs_stock_data.csv"
selected_macro_path = "macro_list.txt"
with open(selected_macro_path, "r") as f:
    Macro_features = [line.strip() for line in f if line.strip()]
macro_data_path = "../../../datasets/macro_processed/macro_data.csv"

stock_data = pd.read_csv(stock_data_path)
stock_data["Date"] = pd.to_datetime(stock_data["Date"])
full_macro_data = pd.read_csv(macro_data_path)
full_macro_data["Date"] = pd.to_datetime(full_macro_data["Date"])

missing_features = [feature for feature in Macro_features if feature not in full_macro_data.columns]
if missing_features:
    raise KeyError(f"Selected macro features missing from macro_data.csv: {missing_features}")

selected_macro_data = full_macro_data[["Date"] + Macro_features].copy()
selected_macro_data = selected_macro_data.sort_values("Date")

if selected_macro_data[Macro_features].isna().any().any():
    missing_counts = selected_macro_data[Macro_features].isna().sum().sort_values(ascending=False)
    raise ValueError(f"Selected macro data contains NaNs before resampling:\n{missing_counts[missing_counts > 0]}")

stock_start = stock_data["Date"].min()
stock_end = stock_data["Date"].max()
macro_start = selected_macro_data["Date"].min()
macro_end = selected_macro_data["Date"].max()
print("ETF stock data", stock_data.shape, stock_start, stock_end)
print("selected macro data", selected_macro_data.shape, macro_start, macro_end)
print("date stamps of macro data", selected_macro_data.head(10))

selected_macro_data.set_index("Date", inplace=True)
selected_macro_data = selected_macro_data.resample("D").ffill()
selected_macro_data = selected_macro_data.loc[(selected_macro_data.index >= max(stock_start, macro_start)) & (selected_macro_data.index <= min(stock_end, macro_end))]

if selected_macro_data.empty:
    raise ValueError("No overlap between ETF stock dates and selected macro dates after no-leak trimming.")
if selected_macro_data[Macro_features].isna().any().any():
    missing_counts = selected_macro_data[Macro_features].isna().sum().sort_values(ascending=False)
    raise ValueError(f"Resampled macro data contains NaNs:\n{missing_counts[missing_counts > 0]}")

print("resampled macro data", selected_macro_data.head(10))
print("resampled macro date range", selected_macro_data.index.min(), selected_macro_data.index.max())

# save the resampled macro data to a csv file
data_folder = "../../../datasets/macro_processed"
if not os.path.exists(data_folder):
    os.makedirs(data_folder)
selected_macro_data.to_csv(os.path.join(data_folder, "macro_data_resampled.csv"))

ETF stock data (81697, 8) 2000-01-03 00:00:00 2023-12-29 00:00:00
selected macro data (298, 47) 2000-03-01 00:00:00 2024-12-01 00:00:00
date stamps of macro data         Date  exp.real.short.rate.5  exp.inflation.5  real.term.prem.5  \
0 2000-03-01               2.215509         3.216578          0.863452   
1 2000-04-01               2.147000         3.150437          0.779137   
2 2000-05-01               2.340100         3.256609          0.886268   
3 2000-06-01               2.213023         3.166773          0.779450   
4 2000-07-01               2.188380         3.140695          0.729120   
5 2000-08-01               2.169926         3.091196          0.650504   
6 2000-09-01               2.095320         3.090670          0.630465   
7 2000-10-01               2.064981         3.060181          0.562281   
8 2000-11-01               2.065086         3.044371          0.541143   
9 2000-12-01               1.822735         2.929985          0.431220   

   inflation.risk.prem.